# nb_01 — Bronze : ingestion des données

Ce notebook ingère les données d’événements, les données de flux et les données de référence provenant des systèmes RH sources. Pour ce laboratoire, la source est Github.

**Prérequis**
- Lakehouse avec schéma `lh_meridian_hr` attaché en tant que lakehouse par défaut.
- `BASE` ci-dessous pointe vers votre dossier GitHub brut `/data`.

Tout est chargé dans le schéma `bronze`.

## Configuration et mise en place de Spark

**Résumé.** Définit l’URL de base de la source brute, le dossier d’atterrissage et le nom du lakehouse ; active V-Order pour Direct Lake ; et s’assure que le schéma `bronze` existe.

<details>
<summary>Détails ligne par ligne</summary>

- `BASE` — le dossier GitHub brut `/data` depuis lequel chaque flux est téléchargé.
- `LANDING = "Files/landing"` — le dossier relatif au lakehouse dans lequel les flux sont déposés.
- `LH = "lh_meridian_hr"` — le nom du lakehouse cible.
- `from pyspark.sql import functions as F` — les fonctions auxiliaires `F.*` utilisées partout.
- `spark.conf.set("spark.sql.parquet.vorder.default", "true")` — active V-Order afin que les fichiers Delta soient optimisés pour les lectures Direct Lake.
- `CREATE SCHEMA IF NOT EXISTS bronze` — s’assure que le schéma Bronze existe.

</details>

In [ ]:
spark.conf.set("spark.sql.parquet.vorder.default", "true")


from pyspark.sql import functions as F

BASE = "https://raw.githubusercontent.com/modamin/datasets/main/hr/data"
LANDING = "Files/landing"
LH = "lh_meridian_hr"

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

print("landing feeds from:", BASE)

## 1. Télécharger les flux dans `Files/landing`

Télécharge les fichiers hébergés depuis le dépôt GitHub brut dans la zone de fichiers du lakehouse. En production, le pipeline dépose ces fichiers.

**Résumé.** Télécharge chaque flux hébergé dans la zone de fichiers du lakehouse, en détectant les pages d’erreur HTML et en validant le JSON avant l’écriture.

<details>
<summary>Détails ligne par ligne</summary>

- `LOCAL = "/lakehouse/default/Files/landing"` + `os.makedirs(..., exist_ok=True)` — le chemin de montage local du dossier d’atterrissage, créé si nécessaire.
- `def fetch(relative_path)` — télécharge un fichier : construit `source_url` à partir de `BASE`, l’ouvre avec `urllib.request.urlopen` et lève une erreur explicite si la requête échoue.
- La vérification `if b"<html" ...` rejette le HTML (signe d’une URL qui n’est pas brute) ; `if relative_path.endswith(".json"): json.loads(content)` valide le JSON avant l’enregistrement.
- `with open(destination, "wb") ...` — écrit les octets dans le dossier d’atterrissage et affiche la taille.
- La boucle `for relative_path in [...]` récupère le flux pay-grid, les taux de change FX et les trois référentiels.

</details>

In [ ]:
import json
import os
import urllib.request

LOCAL = "/lakehouse/default/Files/landing"
os.makedirs(LOCAL, exist_ok=True)

def fetch(relative_path):
    source_url = f"{BASE}/{relative_path}"
    destination = f"{LOCAL}/{os.path.basename(relative_path)}"

    try:

        with urllib.request.urlopen(source_url) as response:

            content = response.read()

    except Exception as error:

        raise RuntimeError(
            f"Could not download {source_url}. Check BASE and repository access."
        ) from error

    if b"<html" in content[:1000].lower():

        raise RuntimeError(
            f"Downloaded HTML instead of data from {source_url}. Check BASE and repository access."
        )

    if relative_path.endswith(".json"):

        json.loads(content)

    with open(destination, "wb") as output_file:

        output_file.write(content)

    print("  ", destination, len(content), "bytes")

relative_file_paths = [
    "feeds/pay_bands_feed.json",
    "feeds/fx_rates.csv",
    "reference/workers.csv",
    "reference/workers_delta.csv",
    "reference/cost_centers.csv",
]

for relative_file_path in relative_file_paths:

    fetch(relative_file_path)

## 2. Flux pay-grid — déplier l’historique imbriqué des bandes

Un objet par (groupe de classification, niveau) avec un tableau `band_history`. Nous le lisons comme un JSON multiligne et appliquons `explode` au tableau pour obtenir une ligne par (groupe, niveau, effective_date). Cette table Bronze est la **source de la dimension de fourchettes de rémunération SCD Type 2** dans le notebook 3.

**Résumé.** Lit le flux JSON multiligne pay-grid et déplie ses tableaux imbriqués en une ligne aplatie par (groupe, niveau, date d’effet), en écrivant `bronze.pay_bands` — la source de la dimension de fourchettes de rémunération SCD2.

<details>
<summary>Détails ligne par ligne</summary>

- `spark.read.option("multiLine", "true").json(pay_bands_path)` — lit le flux comme un JSON multiligne.
- La vérification `if "classifications" not in raw.columns` arrête rapidement le traitement si la structure du fichier est incorrecte (généralement à cause d’un `BASE` erroné).
- `F.explode("classifications")` — une ligne par objet de classification.
- Le second `.select(...)` extrait `group` et `level`, puis `F.explode("c.band_history")` déploie une ligne par bande historique.
- Le dernier `.select(...)` convertit `effective_date`, `band_min/mid/max` et horodate `_ingested_at` avec `F.current_timestamp()`.
- `write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("bronze.pay_bands")` — remplace la table Bronze.
- Le `print` et `filter(... PA / level 3 ...).show()` confirment le nombre de lignes et affichent un aperçu de l’historique d’une classification.

</details>

In [ ]:
pay_bands_path = f"{LANDING}/pay_bands_feed.json"

raw = spark.read.option("multiLine", "true").json(pay_bands_path)

if "classifications" not in raw.columns:
    raise ValueError("Unexpected pay-bands schema. Check BASE and repository access.")

pay_bands = (
    raw.select(F.explode("classifications").alias("c"))
    .select(
        F.col("c.group").alias("classification_group"),
        F.col("c.level").cast("int").alias("classification_level"),
        F.explode("c.band_history").alias("h"),
    )
    .select(
        "classification_group",
        "classification_level",
        F.to_date("h.effective_date").alias("band_effective_date"),
        F.col("h.band_min").cast("double").alias("band_min"),
        F.col("h.band_mid").cast("double").alias("band_mid"),
        F.col("h.band_max").cast("double").alias("band_max"),
        F.current_timestamp().alias("_ingested_at"),
    )
)

(
    pay_bands.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.pay_bands")
)

print(pay_bands.count(), "band-history rows")
(
    pay_bands.filter("classification_group='PA' AND classification_level=3")
    .orderBy("band_effective_date")
    .show(truncate=False)
)

## 3. FX et référentiels (CSV plat → Bronze Delta)

**Résumé.** Définit une petite fonction auxiliaire qui dépose un CSV avec en-tête dans une table Delta avec un horodatage d’ingestion, puis l’applique aux taux FX et aux trois référentiels.

<details>
<summary>Détails ligne par ligne</summary>

- `def land_csv(path, table)` — lit un CSV avec `header=true` et `inferSchema=true`, ajoute `_ingested_at` et remplace la table Delta cible (avec `overwriteSchema`).
- Le `print(...)` indique le nom de la table et le nombre de lignes.
- Les quatre appels `land_csv(...)` déposent `bronze.fx_rates`, `bronze.workers`, `bronze.workers_delta` et `bronze.cost_centers`.

</details>

In [ ]:
def land_csv(path, table):
    df = (
        spark.read.option("header", "true")
        .option("inferSchema", "true")
        .csv(path)
        .withColumn("_ingested_at", F.current_timestamp())
    )
    
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table)
    )

    print(f"{table:28s} {df.count():>7,} rows")


land_csv(f"{LANDING}/fx_rates.csv", "bronze.fx_rates")
land_csv(f"{LANDING}/workers.csv", "bronze.workers")
land_csv(f"{LANDING}/workers_delta.csv", "bronze.workers_delta")
land_csv(f"{LANDING}/cost_centers.csv", "bronze.cost_centers")

## 4. Ingérer les données d’événements

**Résumé.** Charge les données d’événements dans `bronze.workforce_events_raw` ; télécharge les CSV mensuels d’événements depuis GitHub et les charge dans une table Delta.

<details>
<summary>Détails ligne par ligne</summary>

- La boucle `for month in pd.period_range("2021-01", "2025-12", freq="M")` télécharge chaque CSV mensuel avec `urllib.request.urlretrieve` et lève une erreur explicite en cas d’échec.
- `spark.read ... csv(f"{LANDING}/events/*.csv")` ajoute `_source_file` (`input_file_name()`) et `_ingested_at`, puis remplace `bronze.workforce_events_raw`.
- Le `print` final indique le nombre de lignes chargées par la solution de secours.

</details>

In [ ]:
import pandas as pd
import os
import urllib.request

events_local = "/lakehouse/default/Files/landing/events"
os.makedirs(events_local, exist_ok=True)

for month in pd.period_range("2021-01", "2025-12", freq="M").strftime("%Y-%m"):
    source_url = f"{BASE}/events/workforce_events_{month}.csv"
    destination = f"{events_local}/workforce_events_{month}.csv"

    try:
        urllib.request.urlretrieve(source_url, destination)
    except Exception as error:
        raise RuntimeError(
            f"Could not download {source_url}. Check BASE and repository access."
        ) from error

raw = (
    spark.read.option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{LANDING}/events/*.csv")
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingested_at", F.current_timestamp())
)

(
    raw.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.workforce_events_raw")
)

print("Events loaded:", f"{raw.count():,}")

## Lister les tables Bronze

**Résumé.** Affiche les tables actuellement présentes dans le schéma `bronze` comme vérification rapide de fin de traitement.

<details>
<summary>Détails ligne par ligne</summary>

- `spark.sql("SHOW TABLES IN bronze").collect()` — liste les tables Bronze et affiche chaque `tableName`.

</details>

In [ ]:
for t in spark.sql("SHOW TABLES IN bronze").collect():
    print("  ", t.tableName)